# MosesLukeGISmapdeveloper: make a clean, labelled map in your browser

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Moses-L/geomapclean/blob/main/notebooks/geomapclean_colab.ipynb)
[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.24010663.svg)](https://doi.org/10.5281/zenodo.24010663)

**Built by Moses Luke.** This notebook walks you through turning a raw GIS file (a zipped shapefile, GeoJSON, GeoPackage or KML) into a finished map with colours, names that do not overlap, a legend, a scale bar and a north arrow. You do not need to install anything and you do not need to write code. You only change a few settings and press play.

### Who this is for
Researchers, students, epidemiologists and programme teams who have boundary files and numbers and want a tidy figure for a paper, report or slide.

### How long it takes
About 5 minutes the first time. The first cell installs the tool and takes about a minute.

### How to use a notebook, in 30 seconds
- A notebook is a list of **cells**. Grey cells hold code. White cells hold text like this one.
- To run a code cell, click it and press **Shift + Enter**, or click the round play button on its left.
- Run the cells **from top to bottom, one at a time**. Wait until a cell finishes (the play button stops spinning) before running the next.
- Some cells have a small form on the right side. Change the values there. You never need to edit the code itself.

### What you will do
| Step | What happens |
|---|---|
| 1 | Install the tool |
| 2 | Choose your data: the built-in demo, or your own file |
| 3 | Look inside the file to see what it contains |
| 4 | (Optional) Attach your own numbers from a CSV or Excel file |
| 5 | Choose the style and draw the map |
| 6 | Read the report, then download the map |

**Privacy note.** In Colab, your files go to a temporary computer that Google provides for your session. They are not stored by this tool. Even so, do not upload sensitive or identifiable data to any shared online service. For confidential data, run the tool on your own computer instead (see the README on GitHub).

## Step 1. Install the tool

Run the cell below. It installs the tool from PyPI, the public Python package library. You will see some text scroll past. That is normal.

When it finishes, it prints the installed version. If Colab shows a button saying **Restart session**, click it, then run this cell again once.

In [ ]:
!pip install -q moseslukegeomap
import moseslukegeomap
print("Installed geomapclean version", moseslukegeomap.__version__)

## Step 2. Choose your data

You have two choices. Use the form on the right of the next cell.

- **Try the demo (recommended the first time).** Tick `USE_DEMO`. The tool uses an invented island called "Null Island", with 16 districts. Nothing real is shown, so you can experiment freely.
- **Use your own file.** Untick `USE_DEMO`. A **Choose files** button appears. Pick your file, and wait for the upload to reach 100%.

**What to upload**

| You have | What to upload |
|---|---|
| A shapefile | One `.zip` containing the `.shp`, `.shx` and `.dbf` files. Add the `.prj` file too, so the map projection is known. |
| A GeoJSON, GeoPackage (`.gpkg`) or KML file | The file as it is. |
| Several loose shapefile parts | Select all the parts together in the upload window. |

In [ ]:
#@title Choose your data { run: "auto" }
USE_DEMO = True  #@param {type:"boolean"}

from moseslukegeomap import Source, describe, read_table, join_table, filter_rows
from moseslukegeomap.workflow import demo_layer, run

src = None
if USE_DEMO:
    print("Using the built-in demo: an invented island with 16 districts.")
else:
    from google.colab import files
    uploaded = files.upload()          # a window opens: choose your file(s)
    src = Source(list(uploaded.items()))
    print("Layers found in your upload:")
    for i, m in enumerate(src.members):
        print(f"  [{i}] {m}")

### Pick the layer to map

Some files hold several layers, for example states, districts and wards. The list printed above shows what was found, numbered from 0. Type the number of the layer you want in `LAYER_NUMBER`.

If you are using the demo, leave this at 0 and just run the cell.

In [ ]:
#@title Pick the layer { run: "auto" }
LAYER_NUMBER = 0  #@param {type:"integer"}

if USE_DEMO:
    layer = demo_layer()
else:
    chosen = src.members[LAYER_NUMBER]
    print("Reading:", chosen)
    layer = src.read(chosen)

print(f"Loaded {len(layer)} shapes.")

## Step 3. Look inside the file

Before drawing, it helps to know what is in the file. The cell below prints the number of shapes, the type of shape, the map projection, and every column with an example value.

**Why this matters:** you will need to know the **name column** (for labels) and any **number column** (to colour the map by). Note their exact spelling, because the next steps ask for them.

In [ ]:
info = describe(layer)
print("Shapes:         ", info["features"])
print("Shape type:     ", info["geometry_types"])
print("Projection:     ", info["crs"])
print()
print("Columns you can use:")
for c in info["columns"]:
    print(f"  {c['name']:<22} {c['dtype']:<10} example: {c['example']}")
layer.drop(columns="geometry").head()

## Step 4 (optional). Attach your own numbers

Skip this step if the numbers you want to show are already in the file, as they are in the demo (the `coverage_pct` column).

If your numbers are in a separate CSV or Excel file, this step joins them to the shapes by matching area names.

**Your table needs** one row per area, a column holding the area name, and one or more number columns. For example:

| state | coverage |
|---|---|
| Lagos | 82 |
| Kano | 61 |

**How to use this step**
1. Tick `ATTACH_TABLE`.
2. In `MAP_NAME_COLUMN`, type the column in the **map** that holds the area names (from Step 3).
3. In `TABLE_NAME_COLUMN`, type the column in **your table** that holds the same names.
4. Run the cell and upload your table when asked.

The tool matches names even if capital letters, spaces or punctuation differ. It then lists any names that did **not** match, with suggestions. Fix those spellings in your table and run the cell again.

For the demo, ticking `ATTACH_TABLE` creates a small example table so you can see how it works. After a join, the new number columns can be used in Step 5: type the new column name in `COLOR_BY`.

In [ ]:
#@title Attach a table { run: "auto" }
ATTACH_TABLE = False          #@param {type:"boolean"}
MAP_NAME_COLUMN = "district"  #@param {type:"string"}
TABLE_NAME_COLUMN = "district"  #@param {type:"string"}

table = None
if ATTACH_TABLE:
    if USE_DEMO:
        import pandas as pd
        names = list(layer["district"])
        # note the lower-case spelling and one deliberate typo, to show the matching report
        table = pd.DataFrame({
            "district": [n.lower() for n in names[:-1]] + ["Typo district"],
            "target_pct": [60 + (i * 3) % 35 for i in range(len(names))],
        })
    else:
        from google.colab import files
        up = files.upload()
        fname, data = next(iter(up.items()))
        table = read_table(data, fname)

    layer, report = join_table(layer, table, MAP_NAME_COLUMN, TABLE_NAME_COLUMN)
    print("Rows matched:        ", report.get("matched"))
    print("Map names with no row:", report["unmatched_in_map"])
    print("Table rows with no area:", report.get("unmatched_in_table"))
    print("Suggestions:         ", report["suggestions"])
    print("\nNew columns now available:", [c for c in layer.columns if c != "geometry"])
else:
    print("No table attached. The map will use the columns already in the file.")

## Step 5. Choose the style and draw the map

Use the form on the right of the next cell. Every setting has a plain-English meaning:

| Setting | What it does | Tips |
|---|---|---|
| `TITLE`, `SUBTITLE`, `SOURCE` | Text printed on the map | Always credit where your boundary data came from in `SOURCE`. |
| `COLOR_BY` | The column that sets the colours | Type the exact column name from Step 3. Leave empty to draw plain outlines. |
| `CLASSIFICATION` | How values are grouped into colour steps | `quantiles` puts equal numbers of areas in each group. `equal` splits the value range into equal steps. `jenks` finds natural clusters. `categorical` gives each category its own colour (use it for text columns such as zone). |
| `CLASSES` | Number of colour steps | 4 to 6 is easiest to read. |
| `LEGEND_TITLE` | Heading of the legend | Include the unit, for example "Coverage (%)". |
| `LABEL_COLUMN` | The column whose text is written on the map | Leave as `auto` and the tool finds the name column itself. |
| `COLOR_SCALE` | The colour palette | `YlOrRd`, `Blues` and `Greens` print well in black and white. |
| `THEME` | Overall look | `light` for reports, `paper` for a warm tone, `dark` for slides. |
| `WIDTH_INCHES` | Width of the figure | Make it wider to fit more labels on a crowded map. |
| `FOCUS_COLUMN`, `FOCUS_VALUES` | Draw only part of the map | Example: column `State`, values `Lagos`. Separate several values with commas. This is the best way to name every area on a map with hundreds of small areas. |

Change the settings, run the cell, and look at the map. Then change a setting and run it again. You can repeat this as often as you like.

In [ ]:
#@title Style your map { run: "auto" }
TITLE = "Coverage by district"   #@param {type:"string"}
SUBTITLE = "Demo data, invented"  #@param {type:"string"}
SOURCE = "Source: invented demo data"  #@param {type:"string"}
COLOR_BY = "coverage_pct"        #@param {type:"string"}
CLASSIFICATION = "quantiles"     #@param ["auto", "quantiles", "equal", "jenks", "categorical"]
CLASSES = 5                      #@param {type:"slider", min:3, max:8, step:1}
LEGEND_TITLE = "Coverage (%)"    #@param {type:"string"}
LABEL_COLUMN = "auto"            #@param {type:"string"}
COLOR_SCALE = "YlOrRd"           #@param ["YlOrRd", "Blues", "Greens", "Purples", "Oranges", "YlGnBu", "viridis", "magma", "RdBu", "RdYlGn", "Spectral"]
THEME = "light"                  #@param ["light", "paper", "dark"]
WIDTH_INCHES = 10                #@param {type:"slider", min:6, max:16, step:1}
FOCUS_COLUMN = ""                #@param {type:"string"}
FOCUS_VALUES = ""                #@param {type:"string"}

where = None
if FOCUS_COLUMN and FOCUS_VALUES:
    where = (FOCUS_COLUMN, [v.strip() for v in FOCUS_VALUES.split(",") if v.strip()])

options = dict(
    title=TITLE or None, subtitle=SUBTITLE or None, source=SOURCE or None,
    color_by=COLOR_BY or None, classification=CLASSIFICATION, k=CLASSES,
    legend_title=LEGEND_TITLE or None, label_column=LABEL_COLUMN or "auto",
    cmap=COLOR_SCALE, theme=THEME, width_in=WIDTH_INCHES,
)

try:
    result, extra = run(layer, where=where, **options)
except Exception as err:
    print("The map could not be drawn:", err)
    print("Check that COLOR_BY and the focus column match a column name exactly (see Step 3).")
else:
    from IPython.display import Image, display
    display(Image(data=result.to_bytes("png", 150)))

## Step 6. Read the report, then download

The tool tells you what it did, so you can check it before using the map in a paper.

How to read the report:
- **Projection** shows the map projection it chose for your area.
- **Labels placed / skipped** shows how many names were written and how many were left out. On a map with many tiny areas, the tool leaves some names out rather than crowd the map. To name more, use a **focus** (Step 5), make the figure **wider**, or ask for leader lines (see the README).
- **Notes** lists anything it repaired, for example invalid shapes.

In [ ]:
for key, value in result.report.items():
    print(f"{key}: {value}")

### Download your map

The next cell saves the map in three formats and downloads them to your computer:

| Format | Best for |
|---|---|
| **PNG** (300 dpi) | Slides, Word documents, web pages |
| **SVG** | Editing in Inkscape or Illustrator; stays sharp at any size |
| **PDF** | Journals and printing |

If your browser blocks the downloads, allow pop-ups for this page and run the cell again. You can also open the folder icon on the left of Colab, find the files, and download them from there.

In [ ]:
from google.colab import files
for ext, dpi in (("png", 300), ("svg", None), ("pdf", None)):
    path = f"my_map.{ext}"
    result.save(path, dpi=dpi) if dpi else result.save(path)
    files.download(path)
print("Done. Your map files should now be in your Downloads folder.")

## If something goes wrong

| What you see | What to do |
|---|---|
| `No module named moseslukegeomap` | Run Step 1 again. If Colab asks you to restart the session, do so, then run Step 1 again. |
| The upload window does not appear | Make sure `USE_DEMO` is unticked, then run the cell again. |
| "No map layer found" | Your zip is missing `.shp`, `.shx` or `.dbf`. Zip the full set of files together. |
| `KeyError` or "column not found" | A column name is spelled differently from Step 3. Copy it exactly, including capital letters. |
| Some areas show as "No data" | Their names did not match your table. Read the unmatched list in Step 4 and fix the spelling in your table. |
| Many labels are missing | Use a focus (Step 5), or make the figure wider. |
| The cell runs for a very long time | Very large files (hundreds of thousands of shapes) take longer. Focus on one region first. |

Still stuck? Open an issue at https://github.com/Moses-L/geomapclean/issues. Describe the file type, what you expected and what happened. Do not attach confidential data.

## Using the map in your work

1. **Credit your boundary data** in the `SOURCE` line. Boundary files have their own publishers and licences.
2. **Say how you grouped the values** in your figure caption, for example "five quantile classes".
3. **Cite the tool.** All-versions DOI: https://doi.org/10.5281/zenodo.24010663

> Luke, M. (2026). *geomapclean: Clean, labelled maps from raw GIS files* [Computer software]. Zenodo. https://doi.org/10.5281/zenodo.24010663

## About

**MosesLukeGISmapdeveloper** (Python package `moseslukegeomap`, command `geomapclean`) was built by Moses Luke, an epidemiologist and health information specialist in Abuja, Nigeria. It is free and open source under the MIT licence.

- Live web app: https://moseslukegismap.streamlit.app/
- Source code: https://github.com/Moses-L/geomapclean
- Package: https://pypi.org/project/moseslukegeomap/
- ORCID: https://orcid.org/0000-0002-0600-6660